# AURORA SCORE — Prototipo técnico (EP5)
**Gestión de Proyectos de Datos (SCY1102)** · Equipo: Ignacio Cerda, Sebastián González, Jesús Morán

Implementa el tramo F3–F5 del plan de proyecto (CRISP-DM): ingesta batch/streaming, calidad, variables, modelo de score, equidad y explicabilidad.

> ⚠️ **Datos 100% sintéticos.** No son datos de VFG; reproducen los sesgos descritos en el Informe N°1 para poder probar el pipeline sin exponer datos personales.

In [1]:
import sys, warnings; warnings.filterwarnings('ignore')
sys.path.insert(0, '..')
import numpy as np, pandas as pd
from aurora import datagen, pipeline as pl, modelo as md
pd.set_option('display.width', 160)

## 1. Datos y fuentes (RF01)
Seis fuentes: telecomunicaciones, utilities, billeteras/e-commerce, app VFG, redes sociales (opt-in) y biometría (solo antifraude, **fuera del score**).

In [2]:
pop, rng = datagen.generar_poblacion()
fuentes = datagen.generar_fuentes(pop, rng)
hist = datagen.generar_historico_sesgado(pop, rng)
print({k: v.shape for k, v in fuentes.items()})
print('Mora real en la población:', round(pop.mora_real.mean(), 3))

{'telco': (40000, 7), 'utilities': (40000, 4), 'billeteras': (40000, 5), 'app': (40000, 5), 'social': (40000, 4), 'biometria': (40000, 3)}
Mora real en la población: 0.114


## 2. Ingesta batch y streaming (RF01, RNF01)

In [3]:
lote, m_batch = pl.ingesta_batch(fuentes, '../data')
print('BATCH     ', m_batch)
estado, m_stream = pl.ingesta_streaming(pl.generar_eventos(fuentes, 200_000, rng))
print('STREAMING ', {k: round(v, 2) for k, v in m_stream.items()})

BATCH      {'filas': 120000, 'segundos': 0.3177339349999784}
STREAMING  {'eventos': 200000, 'segundos': 0.16, 'eventos_por_seg': 1219912.49}


## 3. Calidad y frescura por fuente (RF03)
Se marca como *score con datos rezagados* a quien supera el SLA de su fuente (telecomunicaciones: 48 h, según el Informe N°1).

In [4]:
calidad = pl.reporte_calidad(fuentes); calidad

,fuente,modo,filas,completitud,duplicados_id,rezago_max_h,pct_fuera_SLA
0,telco,batch,40000,0.962,0,72,0.201
1,utilities,batch,40000,0.970,0,720,0.000
2,billeteras,streaming,40000,0.851,0,0,0.000
3,app,streaming,40000,1.000,0,0,0.000
4,social,batch,40000,0.701,0,24,0.000
5,biometria,streaming,40000,1.000,0,0,0.000


In [5]:
X = pl.construir_features(fuentes)
demo = pop.set_index('id')[['rural','edad','informal']].copy(); demo['adulto_mayor'] = (demo.edad >= 60).astype(int)
print('Variables (sin biometría):', X.shape[1])
print('Completitud telco por grupo (sesgo geográfico):', pl.completitud_por_grupo(fuentes, pop, 'telco', ['regularidad_recarga','antiguedad_telefono_meses']))

Variables (sin biometría): 18
Completitud telco por grupo (sesgo geográfico): {'rural': {0: 0.961, 1: 0.776}, 'adulto_mayor': {0: 0.906, 1: 0.908}, 'informal': {0: 0.914, 1: 0.884}}


## 4. Sesgo de representatividad en el histórico de entrenamiento
El Informe N°1 advirtió que la base histórica sub-representa a población rural, adultos mayores y trabajo informal.

In [6]:
en_hist = hist.set_index('id')['en_historico'].values
y_obs = hist.set_index('id')['mora_obs'].values; y_real = pop.set_index('id')['mora_real'].values
test = np.random.default_rng(11).random(len(X)) < 0.30   # población representativa para evaluar
tr = en_hist & ~test
comp = pd.DataFrame({'población': demo[['rural','adulto_mayor','informal']].mean(), 'histórico (entrenamiento)': demo[tr][['rural','adulto_mayor','informal']].mean()}).round(3)
print('n entrenamiento =', tr.sum(), '| n evaluación =', test.sum()); comp

n entrenamiento = 15230 | n evaluación = 12170


,población,histórico (entrenamiento)
rural,0.300,0.070
adulto_mayor,0.099,0.033
informal,0.280,0.112


## 5. Modelos (RF04) — gradient boosting
* **M0**: línea base, entrenada con el histórico tal cual.
* **M1**: M0 + *reweighting* por grupo.
* **M2**: M1 + retiro de variables proxy (`antiguedad_telefono_meses`, `plan_postpago`, `densidad_antenas`).

In [7]:
res = {}
for nombre, qp, rw in [('M0 base', False, False), ('M1 reweighting', False, True), ('M2 reweighting + sin proxys', True, True)]:
    cols = md.columnas_modelo(X, qp)
    m = md.entrenar(X[cols][tr], y_obs[tr], md.pesos_reweighting(demo[tr]) if rw else None)
    res[nombre] = (m, cols, md.evaluar(m, X[cols][test], y_real[test], demo[test]))
tabla = pd.DataFrame([dict(modelo=k, variables=len(v[1]), AUC=round(v[2]['auc'],3), KS=round(v[2]['ks'],3), razon_min_aprobacion=v[2]['equidad'].razon_aprobacion.min(), mora_en_aprobados=round(v[2]['mora_aprobados'],3), cumple_umbral_080=v[2]['equidad'].razon_aprobacion.min() >= 0.80) for k, v in res.items()])
tabla

,modelo,variables,AUC,KS,razon_min_aprobacion,mora_en_aprobados,cumple_umbral_080
0,M0 base,15,0.708,0.303,0.665,0.065,False
1,M1 reweighting,15,0.704,0.297,0.722,0.065,False
2,M2 reweighting + sin proxys,12,0.707,0.301,0.935,0.064,True


## 6. Auditoría de equidad (RF08, RNF06)
Razón de aprobación = tasa de aprobación del grupo ÷ tasa del resto, con un corte común (se aprueba al 60% de menor riesgo). Umbral de trabajo: **≥ 0,80**.

In [8]:
for k, v in res.items():
    print(k); display(v[2]['equidad'])

M0 base


,grupo,aprob_grupo,aprob_resto,razon_aprobacion,falso_rechazo_grupo,falso_rechazo_resto
0,Rural,0.444,0.667,0.665,0.524,0.299
1,Adulto mayor (60+),0.585,0.602,0.973,0.384,0.365
2,Trabajo informal,0.561,0.615,0.912,0.404,0.352


M1 reweighting


,grupo,aprob_grupo,aprob_resto,razon_aprobacion,falso_rechazo_grupo,falso_rechazo_resto
0,Rural,0.473,0.655,0.722,0.494,0.312
1,Adulto mayor (60+),0.587,0.601,0.976,0.384,0.365
2,Trabajo informal,0.568,0.612,0.927,0.398,0.355


M2 reweighting + sin proxys


,grupo,aprob_grupo,aprob_resto,razon_aprobacion,falso_rechazo_grupo,falso_rechazo_resto
0,Rural,0.572,0.612,0.935,0.392,0.355
1,Adulto mayor (60+),0.576,0.603,0.956,0.393,0.363
2,Trabajo informal,0.586,0.605,0.969,0.379,0.361


## 7. Explicabilidad (RF06)
Importancia global por permutación y explicación local por oclusión (aproximación a SHAP/LIME sin dependencias adicionales).

In [9]:
mF, colsF, rF = res['M2 reweighting + sin proxys']
imp = md.importancia(mF, X[colsF][test], y_real[test]); display(imp.head(10).round(4))
Xt = X[colsF][test]; i = np.where((rF['aprobado']==0) & (demo[test].rural.values==1))[0][0]
base, contrib = md.explicar_local(mF, Xt, Xt.iloc[i])
print(f'Solicitante rural rechazado — prob. de mora = {base:.3f}'); contrib.round(4)

pagos_a_tiempo_12m         0.1230
regularidad_recarga        0.0287
ratio_ingreso_egreso       0.0053
recarga_mensual_usd        0.0021
ticket_promedio_usd        0.0018
recarga_por_tx             0.0012
cambios_dispositivo_6m     0.0012
sesiones_30d               0.0008
antiguedad_cuenta_meses    0.0003
tx_mensuales               0.0001
dtype: float64

Solicitante rural rechazado — prob. de mora = 0.267


pagos_a_tiempo_12m     0.1325
tx_mensuales           0.0254
regularidad_recarga   -0.0166
recarga_mensual_usd   -0.0132
ticket_promedio_usd   -0.0131
dtype: float64

## 8. Latencia de inferencia (RNF01)
⚠️ Mide solo la inferencia en el mismo proceso, **no** la latencia extremo a extremo (red, API, ingesta). La meta de < 8 s se valida en F6.

In [10]:
lat = md.latencia_ms(mF, Xt); {k: round(v, 2) for k, v in lat.items()}

{'p50': 1.4, 'p95': 1.76, 'max': 3.8}

## 9. Conclusiones
* La línea base **no cumple** el criterio de equidad para población rural (razón 0,665); el *reweighting* solo no basta (0,722).
* Retirar las variables proxy cumple el umbral en los tres grupos (≥ 0,935) con una pérdida de AUC despreciable (0,708 → 0,707).
* La biometría queda fuera del score; la calidad por fuente se reporta y los scores con rezago se marcan.
* Pendiente: validar con datos reales de VFG, latencia extremo a extremo y reentrenamiento (MLOps).